In [225]:
import pandas as pd
import numpy as np

In [226]:
# Raw 운동 영상 데이터 불러오기
df_guide = pd.read_csv("data/raw/api_1_guide.csv")
df_video = pd.read_csv("data/raw/api_3_video.csv")
df_mscl = pd.read_csv("data/raw/api_4_mscl.csv")
df_std = pd.read_csv("data/raw/api_5_std.csv")
df_routine = pd.read_csv("data/raw/api_6_routine.csv")
df_all = pd.read_csv("data/raw/api_7_all.csv")

In [227]:
# 운동 영상 데이터 통합에 사용할 변수 정의
selected_columns = [
    # 영상 기본 정보
    "file_nm",
    "vdo_ttl_nm",
    "file_url",
    "vdo_len",
    "vdo_desc",
    "aggrp_nm",

    # 운동 및 체력요인
    "trng_nm",
    "ftns_fctr_nm",
    "ftns_lvl_nm",

    # 운동 부위 및 근육
    "trng_part_nm",
    "trng_mscl_nm",
    "trng_mscl_zn_nm",
    "trng_mscl_part",

    # 운동 방법
    "tool_nm",
    "trng_plc_nm",
    "rptt_tcnt_nm",
    "set_cnt_nm",
    "trng_hr_nm",
    "ecrg_cycl_nm",

    # 프로그램 구성 정보
    "trng_step_nm",
    "trng_week_nm",
    "trng_sqnc_nm",
    "trng_aim_nm",
    "trng_se_nm"
]

In [228]:
# API별 출처를 표시하고 동일한 컬럼 구조로 통일
datasets = {
    "1_guide": df_guide,
    "3_video": df_video,
    "4_mscl": df_mscl,
    "5_std": df_std,
    "6_routine": df_routine
}

standardized_datasets = []

for source_api, df in datasets.items():
    temp = df.reindex(columns=selected_columns).copy()
    temp["source_api"] = source_api
    standardized_datasets.append(temp)

In [229]:
# 형식을 통일한 운동 영상 데이터 하나로 통합
df_integrated = pd.concat(
    standardized_datasets,
    ignore_index=True
)

In [230]:
# 통합 데이터의 크기와 고유 영상 수 확인
print("전체 Row 수:", len(df_integrated))
print("고유 영상 수:", df_integrated["file_nm"].nunique())

print("\nAPI별 고유 영상 수")
print(
    df_integrated.groupby("source_api")["file_nm"]
    .nunique()
)

전체 Row 수: 14619
고유 영상 수: 1051

API별 고유 영상 수
source_api
1_guide      661
3_video      243
4_mscl       114
5_std          8
6_routine     25
Name: file_nm, dtype: int64


In [231]:
# API별 운동명(trng_nm) 데이터 확인
for source_api, group in df_integrated.groupby("source_api"):
    total = len(group)
    missing = group["trng_nm"].isna().sum()
    unique = group["trng_nm"].nunique()

    print(
        f"{source_api}: "
        f"전체 {total}행 / "
        f"운동명 결측 {missing}행 / "
        f"고유 운동명 {unique}개"
    )

1_guide: 전체 8303행 / 운동명 결측 61행 / 고유 운동명 919개
3_video: 전체 1668행 / 운동명 결측 15행 / 고유 운동명 365개
4_mscl: 전체 671행 / 운동명 결측 28행 / 고유 운동명 114개
5_std: 전체 1827행 / 운동명 결측 31행 / 고유 운동명 148개
6_routine: 전체 2150행 / 운동명 결측 93행 / 고유 운동명 436개


In [232]:
# 1번 API를 영상 단위 데이터로 변환
guide_video = (
    df_guide
    .groupby("file_nm", as_index=False)
    .agg(
        title=("vdo_ttl_nm", "first"),
        video_url=("file_url", "first"),
        fitness_factor=("ftns_fctr_nm", "first")
    )
)

guide_video["source_api"] = "1_guide"

In [233]:
# 1번 API의 영상 단위 변환 결과 확인
print("영상 수:", len(guide_video))
print("file_nm 고유값 수:", guide_video["file_nm"].nunique())
print("체력요인 결측:", guide_video["fitness_factor"].isna().sum())

display(guide_video.head(10))

영상 수: 661
file_nm 고유값 수: 661
체력요인 결측: 0


,file_nm,title,video_url,fitness_factor,source_api
0,0AUDLJ08S_00351.mp4,팔굽혀펴기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
1,0AUDLJ08S_00352.mp4,누워서 뒤로 팔굽혀펴기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
2,0AUDLJ08S_00353.mp4,앉아서 엉덩이 들고 이동하기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
3,0AUDLJ08S_00354.mp4,엎드려 팔로 걷기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
4,0AUDLJ08S_00355.mp4,아령 앞으로 들어올리기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
5,0AUDLJ08S_00356.mp4,아령 뒤로 들어올리기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
6,0AUDLJ08S_00357.mp4,의자 잡고 팔 뒤로 굽히기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
7,0AUDLJ08S_00358.mp4,밴드 앞 옆으로 들어올리기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
8,0AUDLJ08S_00359.mp4,밴드 어깨 뒤로 들어올리기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide
9,0AUDLJ08S_00360.mp4,엎드려 상체 들어올리기,http://openapi.kspo.or.kr/web/video/,근력/근지구력,1_guide


In [234]:
# 1번 API의 체력요인별 영상 수 확인
factor_counts = (
    guide_video["fitness_factor"]
    .value_counts(dropna=False)
)

display(factor_counts)

fitness_factor
유연성        227
근력         148
근력/근지구력     98
심폐지구력       42
협응성         37
순발력         35
민첩성         28
민첩성/순발력     17
평형성         12
협응력         11
전신지구력        4
민첩성/성능       1
유산소          1
Name: count, dtype: int64

In [235]:
# 6번 API의 연속된 동일 운동 스냅샷을 하나의 운동 블록으로 변환
routine_exercises = (
    df_routine
    .dropna(subset=["trng_nm"])
    .sort_values(["file_nm", "snap_tm"])
    .copy()
)

routine_exercises["previous_exercise"] = (
    routine_exercises
    .groupby(["file_nm", "trng_se_nm"])["trng_nm"]
    .shift()
)

routine_exercises["new_block"] = (
    routine_exercises["trng_nm"]
    != routine_exercises["previous_exercise"]
)

routine_exercises["exercise_block"] = (
    routine_exercises
    .groupby(["file_nm", "trng_se_nm"])["new_block"]
    .cumsum()
)

routine_exercises = (
    routine_exercises
    .groupby(
        ["file_nm", "trng_se_nm", "exercise_block"],
        as_index=False,
        dropna=False
    )
    .agg(
        title=("vdo_ttl_nm", "first"),
        video_url=("file_url", "first"),
        trng_nm=("trng_nm", "first"),
        fitness_factor=("ftns_fctr_nm", "first"),
        start_time=("snap_tm", "min"),
        end_time=("snap_tm", "max")
    )
)

routine_exercises["source_api"] = "6_routine"

In [236]:
# 6번 API의 운동 블록 변환 결과 확인
print("원본 Row 수:", len(df_routine))
print("변환 후 운동 블록 수:", len(routine_exercises))
print("고유 영상 수:", routine_exercises["file_nm"].nunique())

display(
    routine_exercises[
        routine_exercises["file_nm"] == "0AUDLJ08S_00041.mp4"
    ][
        [
            "trng_se_nm",
            "trng_nm",
            "fitness_factor",
            "start_time",
            "end_time"
        ]
    ]
)

원본 Row 수: 2150
변환 후 운동 블록 수: 493
고유 영상 수: 25


,trng_se_nm,trng_nm,fitness_factor,start_time,end_time
0,본 운동,복부 당기기,근력/근지구력,543.009,543.009
1,본 운동,복부 당기고 한쪽 다리 구부렸다 펴기,근력/근지구력,571.905,583.516
2,본 운동,엉덩이 들어올리기,근력/근지구력,620.286,635.769
3,본 운동,엉덩이 들고 무릎 가슴으로 당기기,근력/근지구력,670.603,681.014
4,본 운동,누워서 팔 다리 뻗기,근력/근지구력,729.029,761.561
5,본 운동,엎드려서 버티기,근력/근지구력,804.004,816.716
6,본 운동,네발기기 자세로 팔 들어 올리기,근력/근지구력,861.227,908.641
7,본 운동,네발기기 자세로 팔/다리 들어올려 균형 잡기,근력/근지구력,939.005,945.011
8,본 운동,옆으로 누워 굽힌다리 들어주기,근력/근지구력,975.008,1002.502
9,본 운동,옆으로 누워 다리 들어주기,근력/근지구력,1032.032,1038.304


In [237]:
# 6번 API의 영상별 체력요인 운동 개수 계산
routine_factor_counts = (
    routine_exercises
    .dropna(subset=["fitness_factor"])
    .groupby(
        ["file_nm", "fitness_factor"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "exercise_count"})
)

In [238]:
# 예시 영상의 체력요인별 운동 개수 확인
display(
    routine_factor_counts[
        routine_factor_counts["file_nm"] == "0AUDLJ08S_00041.mp4"
    ]
)

print(
    "총 운동 블록 수:",
    routine_factor_counts.loc[
        routine_factor_counts["file_nm"] == "0AUDLJ08S_00041.mp4",
        "exercise_count"
    ].sum()
)

,file_nm,fitness_factor,exercise_count
0,0AUDLJ08S_00041.mp4,근력/근지구력,25
1,0AUDLJ08S_00041.mp4,유연성,22


총 운동 블록 수: 47


In [239]:
# 6번 API의 실제 운동 블록 중 체력요인이 없는 경우 확인
missing_factor = routine_exercises[
    routine_exercises["fitness_factor"].isna()
]

print("전체 운동 블록 수:", len(routine_exercises))
print("체력요인 결측 운동 블록 수:", len(missing_factor))
print("체력요인 결측 영상 수:", missing_factor["file_nm"].nunique())

display(
    missing_factor[
        ["file_nm", "title", "trng_se_nm", "trng_nm"]
    ].head(20)
)

전체 운동 블록 수: 493
체력요인 결측 운동 블록 수: 6
체력요인 결측 영상 수: 3


,file_nm,title,trng_se_nm,trng_nm
100,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),본 운동,네로우 스쿼트/팔 벌리기
108,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),본 운동,X니킥
110,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),본 운동,목 돌리기
331,0AUDLJ08S_00053.mp4,PAPS 4-5등급 학생 체력증진 프로그램 1편(유연성 향상 편),학생건강체력평가제도(유연성),눕기(이완)
418,0AUDLJ08S_01012.mp4,물병을 활용한 유산소 전신 근력 운동,유산소 운동,무릎들고 팔 내리기
420,0AUDLJ08S_01012.mp4,물병을 활용한 유산소 전신 근력 운동,유산소 운동,다리 벌리면서 팔 앞으로 뻗기


In [240]:
# 6번 API의 영상별 원본 체력요인 구성 생성
routine_factor_summary = (
    routine_factor_counts
    .pivot(
        index="file_nm",
        columns="fitness_factor",
        values="exercise_count"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

routine_factor_summary.columns.name = None

In [241]:
# 1번 API의 영상별 원본 체력요인 구성 생성
guide_factor_counts = (
    guide_video[
        ["file_nm", "fitness_factor"]
    ]
    .dropna(subset=["fitness_factor"])
    .assign(exercise_count=1)
)

guide_factor_summary = (
    guide_factor_counts
    .pivot(
        index="file_nm",
        columns="fitness_factor",
        values="exercise_count"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

guide_factor_summary.columns.name = None

In [242]:
# 1번과 6번 API의 공식 체력요인 데이터를 하나로 통합
guide_factor_data = guide_factor_counts.copy()
guide_factor_data["source_api"] = "1_guide"
guide_factor_data["count_type"] = "video_factor"

routine_factor_data = routine_factor_counts.copy()
routine_factor_data["source_api"] = "6_routine"
routine_factor_data["count_type"] = "exercise_block"

official_factor_data = pd.concat(
    [
        guide_factor_data,
        routine_factor_data
    ],
    ignore_index=True
)

official_factor_data = official_factor_data[
    [
        "file_nm",
        "source_api",
        "fitness_factor",
        "exercise_count",
        "count_type"
    ]
]

In [243]:
# 공식 체력요인 데이터의 영상 수와 API별 구성을 확인
print("전체 고유 영상 수:", official_factor_data["file_nm"].nunique())

print("\nAPI별 고유 영상 수")
print(
    official_factor_data
    .groupby("source_api")["file_nm"]
    .nunique()
)

print("\n원본 체력요인 종류")
print(
    sorted(
        official_factor_data["fitness_factor"]
        .dropna()
        .unique()
    )
)

전체 고유 영상 수: 686

API별 고유 영상 수
source_api
1_guide      661
6_routine     25
Name: file_nm, dtype: int64

원본 체력요인 종류
['근력', '근력/근지구력', '민첩성', '민첩성/성능', '민첩성/순발력', '순발력', '심폐지구력', '유산소', '유연성', '전신지구력', '평형성', '협응력', '협응성']


In [244]:
# 3·4·5번 API의 추천 활용 가능 변수와 결측률 확인
candidate_columns = [
    "trng_nm",
    "trng_part_nm",
    "trng_mscl_nm",
    "trng_mscl_zn_nm",
    "rptt_tcnt_nm",
    "set_cnt_nm",
    "trng_hr_nm",
    "tool_nm",
    "aggrp_nm"
]

for name, df in {
    "3_video": df_video,
    "4_mscl": df_mscl,
    "5_std": df_std
}.items():

    available = [col for col in candidate_columns if col in df.columns]

    print(f"\n[{name}]")
    print("고유 영상 수:", df["file_nm"].nunique())

    for col in available:
        missing_rate = df[col].isna().mean() * 100
        unique_count = df[col].nunique()

        print(
            f"{col}: "
            f"결측 {missing_rate:.1f}% / "
            f"고유값 {unique_count}개"
        )


[3_video]
고유 영상 수: 243
trng_nm: 결측 0.9% / 고유값 365개
trng_mscl_nm: 결측 10.7% / 고유값 242개
trng_mscl_zn_nm: 결측 10.7% / 고유값 242개
rptt_tcnt_nm: 결측 84.7% / 고유값 14개
set_cnt_nm: 결측 83.9% / 고유값 7개
trng_hr_nm: 결측 85.4% / 고유값 24개
tool_nm: 결측 24.0% / 고유값 42개
aggrp_nm: 결측 0.1% / 고유값 2개

[4_mscl]
고유 영상 수: 114
trng_nm: 결측 4.2% / 고유값 114개
trng_part_nm: 결측 0.0% / 고유값 3개
rptt_tcnt_nm: 결측 83.9% / 고유값 11개
trng_hr_nm: 결측 96.7% / 고유값 8개
tool_nm: 결측 21.2% / 고유값 15개
aggrp_nm: 결측 0.0% / 고유값 1개

[5_std]
고유 영상 수: 8
trng_nm: 결측 1.7% / 고유값 148개
rptt_tcnt_nm: 결측 94.7% / 고유값 5개
set_cnt_nm: 결측 89.5% / 고유값 3개
trng_hr_nm: 결측 92.3% / 고유값 5개
aggrp_nm: 결측 0.0% / 고유값 2개


In [245]:
# 1번 API의 공식 체력요인을 3번 API 운동명에 연결할 수 있는지 확인
guide_exercise_names = set(
    df_guide["trng_nm"]
    .dropna()
    .str.strip()
)

video_exercise_names = set(
    df_video["trng_nm"]
    .dropna()
    .str.strip()
)

common_exercises = guide_exercise_names & video_exercise_names

print("1번 API 고유 운동명:", len(guide_exercise_names))
print("3번 API 고유 운동명:", len(video_exercise_names))
print("동일 운동명:", len(common_exercises))
print(
    "3번 API 운동명 기준 매칭률:",
    f"{len(common_exercises) / len(video_exercise_names) * 100:.1f}%"
)

print("\n동일 운동명 예시")
print(sorted(common_exercises)[:30])

1번 API 고유 운동명: 919
3번 API 고유 운동명: 365
동일 운동명: 142
3번 API 운동명 기준 매칭률: 38.9%

동일 운동명 예시
['거꾸로 누워서 밀기', '걷기', '계단 두 칸씩 두발 뛰기', '계단 두발 뛰기', '계단 뛰어 오르기', '계단 올라갔다 내려오기', '계단 한발 뛰기', '고정한 상태에서 덤벨 들고 팔꿈치 굽히기', '공 잡고 들어올리기', '공에 엎드려서 상체 들어올리기', '네발기기 자세에서 다리 위로 뻗어올리기', '네발기기 자세에서 손바닥으로 바닥밀기', '누워서 공 뒤로 내리고 당기기', '누워서 공들고 몸 기울이기', '누워서 다리 들어올리기', '누워서 덤벨 들어올리기', '누워서 덤벨 모아들기', '누워서 막대 잡고 팔 들어올리기', '누워서 막대 잡고 팔 회전하기', '누워서 밀기', '누워서 엉덩이 들어올리기', '누워서 팔 다리 동시에 들어올리기', '누워서 하늘 자전거', '다리로 짐볼 들어올리기', '달리기', '덤벨 옆으로 들어올리기', '뒤꿈치 들기', '뒤로 당기기', '뒤로 팔 굽혀 펴기', '뛰어 내렸다가 바로 점프하기']


In [246]:
# 1번 API에서 동일 운동명에 여러 체력요인이 연결되는지 확인
guide_exercise_factor = (
    df_guide
    .dropna(subset=["trng_nm", "ftns_fctr_nm"])
    [["trng_nm", "ftns_fctr_nm"]]
    .copy()
)

guide_exercise_factor["trng_nm"] = (
    guide_exercise_factor["trng_nm"].str.strip()
)

guide_exercise_factor = (
    guide_exercise_factor
    .drop_duplicates()
)

factor_count_by_exercise = (
    guide_exercise_factor
    .groupby("trng_nm")["ftns_fctr_nm"]
    .nunique()
)

print(
    "체력요인 1개인 운동:",
    (factor_count_by_exercise == 1).sum()
)

print(
    "체력요인이 여러 개인 운동:",
    (factor_count_by_exercise > 1).sum()
)

display(
    guide_exercise_factor[
        guide_exercise_factor["trng_nm"].isin(
            factor_count_by_exercise[
                factor_count_by_exercise > 1
            ].index
        )
    ]
    .sort_values("trng_nm")
    .head(30)
)

체력요인 1개인 운동: 901
체력요인이 여러 개인 운동: 18


,trng_nm,ftns_fctr_nm
831,걷기,전신지구력
5527,걷기,심폐지구력
5884,뒤로 팔 굽혀 펴기,근력/근지구력
1644,뒤로 팔 굽혀 펴기,근력
992,몸통 들어올리기,근력
6012,몸통 들어올리기,근력/근지구력
7718,몸통 들어올리기-1,근력/근지구력
2437,몸통 들어올리기-1,근력
6206,몸통 옆으로 굽히기,근력/근지구력
1134,몸통 옆으로 굽히기,근력


In [247]:
# 3번 API와 겹치는 운동 중 체력요인을 하나로 확정할 수 있는 운동 확인
stable_exercises = set(
    factor_count_by_exercise[
        factor_count_by_exercise == 1
    ].index
)

matched_stable_exercises = (
    common_exercises & stable_exercises
)

matched_ambiguous_exercises = (
    common_exercises - stable_exercises
)

print("3번 API 고유 운동명:", len(video_exercise_names))
print("1번 API와 동일한 운동명:", len(common_exercises))
print("체력요인 직접 매핑 가능:", len(matched_stable_exercises))
print("체력요인 충돌:", len(matched_ambiguous_exercises))

print(
    "안전한 직접 매핑률:",
    f"{len(matched_stable_exercises) / len(video_exercise_names) * 100:.1f}%"
)

print("\n충돌 운동명")
print(sorted(matched_ambiguous_exercises))

3번 API 고유 운동명: 365
1번 API와 동일한 운동명: 142
체력요인 직접 매핑 가능: 131
체력요인 충돌: 11
안전한 직접 매핑률: 35.9%

충돌 운동명
['걷기', '뒤로 팔 굽혀 펴기', '몸통 들어올리기', '몸통 옆으로 굽히기', '앉아서 다리 펴기', '앉아서 당겨 내리기', '앉아서 몸통 움츠리기', '앉았다 일어서기', '윗몸 말아 올리기', '팔 굽혀 펴기', '팔 벌려 뛰기']


In [248]:
# 3번 API의 연속된 동일 운동 스냅샷을 하나의 운동 블록으로 변환
video_exercises = (
    df_video
    .dropna(subset=["trng_nm"])
    .sort_values(["file_nm", "snap_tm"])
    .copy()
)

video_exercises["previous_exercise"] = (
    video_exercises
    .groupby("file_nm")["trng_nm"]
    .shift()
)

video_exercises["new_block"] = (
    video_exercises["trng_nm"]
    != video_exercises["previous_exercise"]
)

video_exercises["exercise_block"] = (
    video_exercises
    .groupby("file_nm")["new_block"]
    .cumsum()
)

video_exercises = (
    video_exercises
    .groupby(
        ["file_nm", "exercise_block"],
        as_index=False
    )
    .agg(
        title=("vdo_ttl_nm", "first"),
        video_url=("file_url", "first"),
        trng_nm=("trng_nm", "first"),
        start_time=("snap_tm", "min"),
        end_time=("snap_tm", "max")
    )
)

video_exercises["source_api"] = "3_video"

In [249]:
# 1번 API의 일관된 운동명-체력요인을 3번 API 운동 블록에 매칭
stable_factor_map = (
    guide_exercise_factor[
        guide_exercise_factor["trng_nm"].isin(stable_exercises)
    ]
    .set_index("trng_nm")["ftns_fctr_nm"]
    .to_dict()
)

video_exercises["fitness_factor"] = (
    video_exercises["trng_nm"]
    .str.strip()
    .map(stable_factor_map)
)

video_exercises["fitness_factor_source"] = np.where(
    video_exercises["fitness_factor"].notna(),
    "matched",
    "unavailable"
)

In [250]:
# 3번 API의 영상별 체력요인 매칭 상태 분류
video_match_status = (
    video_exercises
    .groupby("file_nm")
    .agg(
        exercise_count=("trng_nm", "count"),
        matched_count=("fitness_factor", "count")
    )
    .reset_index()
)

video_match_status["match_status"] = np.select(
    [
        video_match_status["matched_count"]
        == video_match_status["exercise_count"],

        video_match_status["matched_count"] > 0
    ],
    [
        "full",
        "partial"
    ],
    default="unavailable"
)

In [251]:
# 3번 API의 완전 매칭 영상별 체력요인 운동 블록 개수 계산
full_video_ids = set(
    video_match_status.loc[
        video_match_status["match_status"] == "full",
        "file_nm"
    ]
)

video_factor_counts = (
    video_exercises[
        video_exercises["file_nm"].isin(full_video_ids)
    ]
    .groupby(
        ["file_nm", "fitness_factor"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "exercise_count"})
)

video_factor_counts["source_api"] = "3_video"
video_factor_counts["fitness_factor_source"] = "matched"
video_factor_counts["count_type"] = "exercise_block"

In [252]:
# 1번, 6번, 3번 API를 합친 전체 추천 후보 영상 수 확인
all_candidate_video_ids = set(official_factor_data["file_nm"]) | set(video_factor_counts["file_nm"])

print("API 1 + API 6 추천 후보:", official_factor_data["file_nm"].nunique())
print("API 3 추가 추천 후보:", video_factor_counts["file_nm"].nunique())
print("전체 추천 후보:", len(all_candidate_video_ids))

API 1 + API 6 추천 후보: 686
API 3 추가 추천 후보: 110
전체 추천 후보: 796


In [253]:
# 1번, 6번, 3번 API의 추천 후보 체력요인 데이터를 하나로 통합
official_candidate_data = official_factor_data.copy()
official_candidate_data["fitness_factor_source"] = "official"

matched_candidate_data = video_factor_counts[
    [
        "file_nm",
        "source_api",
        "fitness_factor",
        "exercise_count",
        "count_type",
        "fitness_factor_source"
    ]
].copy()

candidate_factor_data = pd.concat(
    [
        official_candidate_data,
        matched_candidate_data
    ],
    ignore_index=True
)

candidate_factor_data = candidate_factor_data[
    [
        "file_nm",
        "source_api",
        "fitness_factor",
        "exercise_count",
        "count_type",
        "fitness_factor_source"
    ]
]

In [254]:
# 통합된 추천 후보 영상 수와 API별 영상 수 확인
print("전체 추천 후보:", candidate_factor_data["file_nm"].nunique())

print("\nAPI별 추천 후보")
print(
    candidate_factor_data
    .groupby("source_api")["file_nm"]
    .nunique()
)

print("\n체력요인 출처")
print(
    candidate_factor_data["fitness_factor_source"].value_counts()
)

display(candidate_factor_data.head(20))

전체 추천 후보: 796

API별 추천 후보
source_api
1_guide      661
3_video      110
6_routine     25
Name: file_nm, dtype: int64

체력요인 출처
fitness_factor_source
official    702
matched     110
Name: count, dtype: int64


,file_nm,source_api,fitness_factor,exercise_count,count_type,fitness_factor_source
0,0AUDLJ08S_00351.mp4,1_guide,근력/근지구력,1,video_factor,official
1,0AUDLJ08S_00352.mp4,1_guide,근력/근지구력,1,video_factor,official
2,0AUDLJ08S_00353.mp4,1_guide,근력/근지구력,1,video_factor,official
3,0AUDLJ08S_00354.mp4,1_guide,근력/근지구력,1,video_factor,official
4,0AUDLJ08S_00355.mp4,1_guide,근력/근지구력,1,video_factor,official
5,0AUDLJ08S_00356.mp4,1_guide,근력/근지구력,1,video_factor,official
6,0AUDLJ08S_00357.mp4,1_guide,근력/근지구력,1,video_factor,official
7,0AUDLJ08S_00358.mp4,1_guide,근력/근지구력,1,video_factor,official
8,0AUDLJ08S_00359.mp4,1_guide,근력/근지구력,1,video_factor,official
9,0AUDLJ08S_00360.mp4,1_guide,근력/근지구력,1,video_factor,official


In [255]:
# 전체 추천 후보의 원본 체력요인별 영상 수와 운동 블록 수 확인
factor_summary = (
    candidate_factor_data
    .groupby("fitness_factor")
    .agg(
        video_count=("file_nm", "nunique"),
        total_count=("exercise_count", "sum")
    )
    .sort_values("video_count", ascending=False)
)

display(factor_summary)

,video_count,total_count
fitness_factor,,
유연성,254,441
근력,222,233
근력/근지구력,116,314
심폐지구력,47,52
협응성,37,37
민첩성/순발력,36,88
순발력,35,35
민첩성,28,28
평형성,15,24


In [256]:
# 원본 체력요인을 최종 6개 체력요인 기여도로 변환
factor_mapping = {
    "근력": {
        "strength": 1.0
    },
    "근력/근지구력": {
        "strength": 0.5,
        "muscularEndurance": 0.5
    },
    "심폐지구력": {
        "cardiovascularEndurance": 1.0
    },
    "전신지구력": {
        "cardiovascularEndurance": 1.0
    },
    "유산소": {
        "cardiovascularEndurance": 1.0
    },
    "유연성": {
        "flexibility": 1.0
    },
    "민첩성": {
        "agility": 1.0
    },
    "민첩성/순발력": {
        "agility": 0.5,
        "power": 0.5
    },
    "순발력": {
        "power": 1.0
    }
}

In [257]:
# 추천 후보의 원본 체력요인을 6개 체력요인 기여량으로 변환
fitness_columns = [
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power"
]

factor_contributions = candidate_factor_data.copy()

for column in fitness_columns:
    factor_contributions[column] = 0.0

for index, row in factor_contributions.iterrows():
    mapping = factor_mapping.get(row["fitness_factor"], {})

    for column, weight in mapping.items():
        factor_contributions.loc[index, column] = (
            row["exercise_count"] * weight
        )

In [258]:
# 6개 체력요인 기여량 변환 결과 확인
display(
    factor_contributions[
        [
            "file_nm",
            "fitness_factor",
            "exercise_count",
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power"
        ]
    ].head(30)
)

,file_nm,fitness_factor,exercise_count,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power
0,0AUDLJ08S_00351.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
1,0AUDLJ08S_00352.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
2,0AUDLJ08S_00353.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
3,0AUDLJ08S_00354.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
4,0AUDLJ08S_00355.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
5,0AUDLJ08S_00356.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
6,0AUDLJ08S_00357.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
7,0AUDLJ08S_00358.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
8,0AUDLJ08S_00359.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0
9,0AUDLJ08S_00360.mp4,근력/근지구력,1,0.5,0.5,0.0,0.0,0.0,0.0


In [259]:
# 영상별 6개 체력요인 기여량을 하나의 행으로 통합
video_fitness = (
    factor_contributions
    .groupby("file_nm", as_index=False)[fitness_columns]
    .sum()
)

In [260]:
# 최종 6개 체력요인 기여량이 모두 0인 영상 확인
video_fitness["total_contribution"] = (
    video_fitness[fitness_columns].sum(axis=1)
)

zero_weight_videos = video_fitness[
    video_fitness["total_contribution"] == 0
]

print("전체 후보 영상:", len(video_fitness))
print("6개 체력요인 변환 가능:", (video_fitness["total_contribution"] > 0).sum())
print("6개 체력요인 변환 불가:", len(zero_weight_videos))

display(zero_weight_videos.head(20))

전체 후보 영상: 796
6개 체력요인 변환 가능: 731
6개 체력요인 변환 불가: 65


,file_nm,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,total_contribution
72,0AUDLJ08S_00260.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
73,0AUDLJ08S_00262.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86,0AUDLJ08S_00285.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
112,0AUDLJ08S_00328.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,0AUDLJ08S_00448.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
224,0AUDLJ08S_00449.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
225,0AUDLJ08S_00450.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226,0AUDLJ08S_00451.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
227,0AUDLJ08S_00452.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
228,0AUDLJ08S_00453.mp4,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [261]:
# 6개 체력요인으로 추천 가능한 영상만 선별하고 비율 가중치 계산
video_fitness_weights = (
    video_fitness[
        video_fitness["total_contribution"] > 0
    ]
    .copy()
)

for column in fitness_columns:
    video_fitness_weights[column] = (
        video_fitness_weights[column]
        / video_fitness_weights["total_contribution"]
    )

In [262]:
# 영상별 fitnessWeights 개수와 가중치 합 확인
video_fitness_weights["weight_sum"] = (
    video_fitness_weights[fitness_columns].sum(axis=1)
)

print("최종 추천 후보 영상:", len(video_fitness_weights))

print(
    "가중치 합이 1인 영상:",
    np.isclose(
        video_fitness_weights["weight_sum"],
        1.0
    ).sum()
)

print(
    "가중치 합이 1이 아닌 영상:",
    (~np.isclose(
        video_fitness_weights["weight_sum"],
        1.0
    )).sum()
)

display(video_fitness_weights.head(20))

최종 추천 후보 영상: 731
가중치 합이 1인 영상: 731
가중치 합이 1이 아닌 영상: 0


,file_nm,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,total_contribution,weight_sum
0,0AUDLJ08S_00041.mp4,0.265957,0.265957,0.0,0.468085,0.000000,0.000000,47.0,1.0
1,0AUDLJ08S_00042.mp4,0.250000,0.250000,0.0,0.000000,0.250000,0.250000,10.0,1.0
2,0AUDLJ08S_00043.mp4,0.279070,0.279070,0.0,0.441860,0.000000,0.000000,43.0,1.0
3,0AUDLJ08S_00044.mp4,0.195652,0.195652,0.0,0.608696,0.000000,0.000000,23.0,1.0
4,0AUDLJ08S_00045.mp4,0.176471,0.176471,0.0,0.647059,0.000000,0.000000,34.0,1.0
5,0AUDLJ08S_00046.mp4,0.312500,0.312500,0.0,0.375000,0.000000,0.000000,16.0,1.0
6,0AUDLJ08S_00047.mp4,0.500000,0.500000,0.0,0.000000,0.000000,0.000000,12.0,1.0
7,0AUDLJ08S_00048.mp4,0.483333,0.483333,0.0,0.033333,0.000000,0.000000,30.0,1.0
8,0AUDLJ08S_00049.mp4,0.000000,0.000000,0.0,1.000000,0.000000,0.000000,20.0,1.0
9,0AUDLJ08S_00050.mp4,0.500000,0.500000,0.0,0.000000,0.000000,0.000000,34.0,1.0


In [263]:
# 7번 API의 영상 기본정보를 영상당 하나의 행으로 정리
video_metadata = (
    df_all
    .groupby("file_nm", as_index=False)
    .agg(
        title=("vdo_ttl_nm", "first"),
        file_url=("file_url", "first"),
        video_length=("vdo_len", "first"),
        age_group=("aggrp_nm", "first")
    )
)

In [264]:
# 추천 영상의 fitnessWeights와 영상 기본정보 결합
workout_videos = (
    video_fitness_weights
    .merge(
        video_metadata,
        on="file_nm",
        how="left"
    )
)

workout_videos = workout_videos[
    [
        "file_nm",
        "title",
        "file_url",
        "video_length",
        "age_group",
        "strength",
        "muscularEndurance",
        "cardiovascularEndurance",
        "flexibility",
        "agility",
        "power"
    ]
]

In [265]:
# 추천 영상의 기본정보 결합 여부와 결측값 확인
print("최종 영상 수:", len(workout_videos))
print("고유 영상 수:", workout_videos["file_nm"].nunique())

print("\n기본정보 결측")
print(
    workout_videos[
        ["title", "file_url", "video_length", "age_group"]
    ].isna().sum()
)

display(workout_videos.head(10))

최종 영상 수: 731
고유 영상 수: 731

기본정보 결측
title           0
file_url        0
video_length    0
age_group       0
dtype: int64


,file_nm,title,file_url,video_length,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power
0,0AUDLJ08S_00041.mp4,요통 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,2299,공통,0.265957,0.265957,0.0,0.468085,0.00,0.00
1,0AUDLJ08S_00042.mp4,낙상 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,981,공통,0.250000,0.250000,0.0,0.000000,0.25,0.25
2,0AUDLJ08S_00043.mp4,우울증 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,2928,공통,0.279070,0.279070,0.0,0.441860,0.00,0.00
3,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),http://openapi.kspo.or.kr/web/video/,371,공통,0.195652,0.195652,0.0,0.608696,0.00,0.00
4,0AUDLJ08S_00045.mp4,고혈압 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,2056,공통,0.176471,0.176471,0.0,0.647059,0.00,0.00
5,0AUDLJ08S_00046.mp4,당뇨병 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,1098,공통,0.312500,0.312500,0.0,0.375000,0.00,0.00
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,830,공통,0.500000,0.500000,0.0,0.000000,0.00,0.00
7,0AUDLJ08S_00048.mp4,인지노쇠 예방 운동프로그램,http://openapi.kspo.or.kr/web/video/,1175,공통,0.483333,0.483333,0.0,0.033333,0.00,0.00
8,0AUDLJ08S_00049.mp4,짐볼을 활용한 스트레칭 프로그램,http://openapi.kspo.or.kr/web/video/,841,공통,0.000000,0.000000,0.0,1.000000,0.00,0.00
9,0AUDLJ08S_00050.mp4,짐볼을 활용한 근력운동 프로그램,http://openapi.kspo.or.kr/web/video/,1224,공통,0.500000,0.500000,0.0,0.000000,0.00,0.00


In [266]:
# 추천에 사용하는 API 1·3·6에서 영상별 운동도구 종류 수 확인
equipment_files = {
    "API1": "data/raw/api_1_guide.csv",
    "API3": "data/raw/api_3_video.csv",
    "API6": "data/raw/api_6_routine.csv"
}

for api_name, path in equipment_files.items():
    temp_df = pd.read_csv(path)

    tool_check = (
        temp_df
        .groupby("file_nm")["tool_nm"]
        .agg(
            lambda x: sorted(
                set(x.dropna().astype(str))
            )
        )
    )

    print(f"\n===== {api_name} =====")
    print("전체 영상 수:", len(tool_check))
    print("도구 정보 없는 영상 수:", (tool_check.apply(len) == 0).sum())
    print("도구 1종인 영상 수:", (tool_check.apply(len) == 1).sum())
    print("도구 2종 이상인 영상 수:", (tool_check.apply(len) >= 2).sum())

    print("\n다중 도구 예시")
    display(
        tool_check[
            tool_check.apply(len) >= 2
        ].head(10)
    )


===== API1 =====
전체 영상 수: 661
도구 정보 없는 영상 수: 113
도구 1종인 영상 수: 486
도구 2종 이상인 영상 수: 62

다중 도구 예시


file_nm
0AUDLJ08S_00356.mp4    [물병, 아령]
0AUDLJ08S_00358.mp4    [밴드, 아령]
0AUDLJ08S_00365.mp4    [매트, 밴드]
0AUDLJ08S_00368.mp4    [매트, 밴드]
0AUDLJ08S_00378.mp4    [공, 써클링]
0AUDLJ08S_00419.mp4    [매트, 밴드]
0AUDLJ08S_00424.mp4    [매트, 밴드]
0AUDLJ08S_00490.mp4    [덤벨, 바벨]
0AUDLJ08S_00497.mp4    [덤벨, 바벨]
0AUDLJ08S_00504.mp4    [덤벨, 바벨]
Name: tool_nm, dtype: object


===== API3 =====
전체 영상 수: 243
도구 정보 없는 영상 수: 40
도구 1종인 영상 수: 185
도구 2종 이상인 영상 수: 18

다중 도구 예시


file_nm
0AUDLJ08S_00231.mp4    [덤벨, 물병, 밴드]
0AUDLJ08S_00238.mp4       [매트, 테이블]
0AUDLJ08S_00243.mp4    [매트, 소파, 짐볼]
0AUDLJ08S_00248.mp4        [매트, 짐볼]
0AUDLJ08S_00250.mp4       [짐볼, 테이블]
0AUDLJ08S_00253.mp4        [물병, 밴드]
0AUDLJ08S_00258.mp4      [봉, 봉(의자)]
0AUDLJ08S_00265.mp4       [의자, 테이블]
0AUDLJ08S_00266.mp4        [매트, 의자]
0AUDLJ08S_00268.mp4       [의자, 테이블]
Name: tool_nm, dtype: object


===== API6 =====
전체 영상 수: 25
도구 정보 없는 영상 수: 2
도구 1종인 영상 수: 14
도구 2종 이상인 영상 수: 9

다중 도구 예시


file_nm
0AUDLJ08S_00041.mp4             [매트, 짐볼]
0AUDLJ08S_00042.mp4           [의자, 줄사다리]
0AUDLJ08S_00043.mp4    [공, 스텝박스, 짐볼, 풍선]
0AUDLJ08S_00045.mp4     [매트, 밴드, 의자, 짐볼]
0AUDLJ08S_00047.mp4         [매트, 밴드, 짐볼]
0AUDLJ08S_00054.mp4             [매트, 짐볼]
0AUDLJ08S_00055.mp4         [스텝박스, 줄사다리]
0AUDLJ08S_00056.mp4             [물병, 의자]
0AUDLJ08S_00058.mp4           [줄사다리, 짐볼]
Name: tool_nm, dtype: object

In [267]:
# API 1·3·6의 운동도구 정보를 통합해 영상별 equipment 목록 생성
equipment_data = pd.concat(
    [
        pd.read_csv("data/raw/api_1_guide.csv")[["file_nm", "tool_nm"]],
        pd.read_csv("data/raw/api_3_video.csv")[["file_nm", "tool_nm"]],
        pd.read_csv("data/raw/api_6_routine.csv")[["file_nm", "tool_nm"]]
    ],
    ignore_index=True
)

video_equipment = (
    equipment_data
    .groupby("file_nm")["tool_nm"]
    .agg(
        lambda x: sorted(
            set(x.dropna().astype(str))
        )
    )
    .reset_index(name="equipment")
)

display(video_equipment.head())

,file_nm,equipment
0,0AUDLJ08S_00041.mp4,"[매트, 짐볼]"
1,0AUDLJ08S_00042.mp4,"[의자, 줄사다리]"
2,0AUDLJ08S_00043.mp4,"[공, 스텝박스, 짐볼, 풍선]"
3,0AUDLJ08S_00044.mp4,[]
4,0AUDLJ08S_00045.mp4,"[매트, 밴드, 의자, 짐볼]"


In [268]:
# 영상 기본정보에 영상별 운동도구 목록 결합
video_metadata = video_metadata.merge(
    video_equipment,
    on="file_nm",
    how="left"
)

video_metadata["equipment"] = (
    video_metadata["equipment"]
    .apply(
        lambda x: x
        if isinstance(x, list)
        else []
    )
)

display(
    video_metadata[
        [
            "file_nm",
            "title",
            "age_group",
            "equipment"
        ]
    ].head()
)

,file_nm,title,age_group,equipment
0,0AUDLJ08S_00001.mp4,신장,공통,[]
1,0AUDLJ08S_00002.mp4,허리둘레,공통,[]
2,0AUDLJ08S_00003.mp4,체성분검사,공통,[]
3,0AUDLJ08S_00004.mp4,상대악력,공통,[]
4,0AUDLJ08S_00005.mp4,반복점프,공통,[]


In [269]:
# 최종 731개 추천 영상에 운동도구 정보 결합
saved_workout_videos = saved_workout_videos.merge(
    video_equipment,
    on="file_nm",
    how="left"
)

saved_workout_videos["equipment"] = (
    saved_workout_videos["equipment"]
    .apply(
        lambda x: x
        if isinstance(x, list)
        else []
    )
)

print("최종 영상 수:", len(saved_workout_videos))
print("최종 컬럼 수:", len(saved_workout_videos.columns))

display(
    saved_workout_videos[
        ["file_nm", "title", "equipment"]
    ].head(20)
)

최종 영상 수: 731
최종 컬럼 수: 12


,file_nm,title,equipment
0,0AUDLJ08S_00041.mp4,요통 예방 운동프로그램,"[매트, 짐볼]"
1,0AUDLJ08S_00042.mp4,낙상 예방 운동프로그램,"[의자, 줄사다리]"
2,0AUDLJ08S_00043.mp4,우울증 예방 운동프로그램,"[공, 스텝박스, 짐볼, 풍선]"
3,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),[]
4,0AUDLJ08S_00045.mp4,고혈압 예방 운동프로그램,"[매트, 밴드, 의자, 짐볼]"
5,0AUDLJ08S_00046.mp4,당뇨병 예방 운동프로그램,[매트]
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,"[매트, 밴드, 짐볼]"
7,0AUDLJ08S_00048.mp4,인지노쇠 예방 운동프로그램,[공]
8,0AUDLJ08S_00049.mp4,짐볼을 활용한 스트레칭 프로그램,[짐볼]
9,0AUDLJ08S_00050.mp4,짐볼을 활용한 근력운동 프로그램,[짐볼]


In [270]:
# 최종 추천 영상에 영상별 운동도구 정보 결합
workout_videos = workout_videos.drop(
    columns=["equipment"],
    errors="ignore"
)

workout_videos = workout_videos.merge(
    video_equipment,
    on="file_nm",
    how="left"
)

workout_videos["equipment"] = (
    workout_videos["equipment"]
    .apply(
        lambda x: x
        if isinstance(x, list)
        else []
    )
)

print("최종 영상 수:", len(workout_videos))
print("최종 컬럼 수:", len(workout_videos.columns))

display(
    workout_videos[
        ["file_nm", "title", "equipment"]
    ].head(20)
)

최종 영상 수: 731
최종 컬럼 수: 12


,file_nm,title,equipment
0,0AUDLJ08S_00041.mp4,요통 예방 운동프로그램,"[매트, 짐볼]"
1,0AUDLJ08S_00042.mp4,낙상 예방 운동프로그램,"[의자, 줄사다리]"
2,0AUDLJ08S_00043.mp4,우울증 예방 운동프로그램,"[공, 스텝박스, 짐볼, 풍선]"
3,0AUDLJ08S_00044.mp4,우울증 예방 운동프로그램(댄스운동 편),[]
4,0AUDLJ08S_00045.mp4,고혈압 예방 운동프로그램,"[매트, 밴드, 의자, 짐볼]"
5,0AUDLJ08S_00046.mp4,당뇨병 예방 운동프로그램,[매트]
6,0AUDLJ08S_00047.mp4,골다공증 예방 운동프로그램,"[매트, 밴드, 짐볼]"
7,0AUDLJ08S_00048.mp4,인지노쇠 예방 운동프로그램,[공]
8,0AUDLJ08S_00049.mp4,짐볼을 활용한 스트레칭 프로그램,[짐볼]
9,0AUDLJ08S_00050.mp4,짐볼을 활용한 근력운동 프로그램,[짐볼]


In [271]:
# 영상 기본 경로와 파일명을 결합해 실제 재생 가능한 영상 URL 생성
def build_video_url(file_url, file_nm):

    file_url = str(file_url).rstrip("/")
    file_nm = str(file_nm)

    if file_url.endswith(file_nm):
        return file_url

    return f"{file_url}/{file_nm}"


workout_videos["file_url"] = workout_videos.apply(
    lambda row: build_video_url(
        row["file_url"],
        row["file_nm"]
    ),
    axis=1
)

url_ends_with_file_nm = workout_videos.apply(
    lambda row: row["file_url"].endswith(
        row["file_nm"]
    ),
    axis=1
)

duplicated_file_nm = workout_videos.apply(
    lambda row: row["file_url"].count(
        row["file_nm"]
    ) >= 2,
    axis=1
)

print("workout_videos 행 수:", len(workout_videos))
print(
    "file_url이 file_nm으로 끝나는 영상 수:",
    int(url_ends_with_file_nm.sum())
)
print(
    "file_nm이 URL에 두 번 이상 붙은 행:",
    int(duplicated_file_nm.sum())
)

print("\n예시 영상 3개")
for _, row in workout_videos.head(3).iterrows():
    print(row["file_nm"], row["file_url"])

workout_videos 행 수: 731
file_url이 file_nm으로 끝나는 영상 수: 731
file_nm이 URL에 두 번 이상 붙은 행: 0

예시 영상 3개
0AUDLJ08S_00041.mp4 http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00041.mp4
0AUDLJ08S_00042.mp4 http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00042.mp4
0AUDLJ08S_00043.mp4 http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00043.mp4


In [272]:
# equipment가 포함된 최종 추천 영상 데이터를 CSV 파일로 저장
workout_videos.to_csv(
    "data/processed/workout_videos.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "CSV 저장 완료:",
    workout_videos.shape
)

CSV 저장 완료: (731, 12)


In [273]:
# 최종 추천 영상 데이터를 개발 연동용 JSON 구조로 변환
workout_video_json = []

for _, row in workout_videos.iterrows():

    fitness_weights = {
        factor: float(row[factor])
        for factor in [
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power"
        ]
        if row[factor] > 0
    }

    workout_video_json.append({
        "videoId": row["file_nm"],
        "title": row["title"],
        "videoUrl": row["file_url"],
        "ageGroup": row["age_group"],
        "equipment": row["equipment"],
        "fitnessWeights": fitness_weights
    })

print(
    "JSON 변환 데이터 수:",
    len(workout_video_json)
)

JSON 변환 데이터 수: 731


In [274]:
# 개발 연동용 최종 추천 영상 데이터를 JSON 파일로 저장
import json

with open(
    "data/processed/workout_videos.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        workout_video_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "JSON 저장 완료:",
    len(workout_video_json)
)

JSON 저장 완료: 731
